## Import Libraries

In [4]:
import requests
import pandas as pd

## Make the API request

In [14]:
# API endpoint that provies weather forecast data.
url = "https://api.open-meteo.com/v1/forecast"

# Parameters tell the API exactly what data we want.
params = {
    "latitude": 47.6101,
    "longitude": -122.2015,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "America/Los_Angeles"
}

# Send a GET request to the API and store the response.
response = requests.get(url, params=params)

# Display the HTTP status code (200 means the request was successful).
response.status_code

200

## Inspect the data we received

In [15]:
 # Convert the API response into a python dictionary.
weather_data = response.json()

# Display the raw data returned by the API.
weather_data

{'latitude': 47.616787,
 'longitude': -122.210236,
 'generationtime_ms': 0.087738037109375,
 'utc_offset_seconds': -25200,
 'timezone': 'America/Los_Angeles',
 'timezone_abbreviation': 'GMT-7',
 'elevation': 27.0,
 'daily_units': {'time': 'iso8601',
  'temperature_2m_max': '°C',
  'temperature_2m_min': '°C',
  'precipitation_sum': 'mm'},
 'daily': {'time': ['2026-07-07',
   '2026-07-08',
   '2026-07-09',
   '2026-07-10',
   '2026-07-11',
   '2026-07-12',
   '2026-07-13'],
  'temperature_2m_max': [24.6, 22.3, 25.8, 22.4, 26.3, 24.3, 30.8],
  'temperature_2m_min': [13.5, 14.0, 12.3, 12.2, 12.1, 9.8, 10.2],
  'precipitation_sum': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}}

## Save Raw JSON

In [16]:
import json
# Save the raw API response as a JSON file.
with open('../data/raw/bellevue_weather_raw.json', 'w') as file:
    json.dump(weather_data, file, indent=4)

## Convert Json into a table

In [7]:
# Pull out the daily weather section from the full API response.
daily_data = weather_data["daily"]

# Convert the daily weather dictionary into a pandas DataFrame.
weather_df = pd.DataFrame(daily_data)

# Display the first few rows of the DataFrame
weather_df.head()

,time,temperature_2m_max,temperature_2m_min,precipitation_sum
0,2026-07-07,24.6,13.5,0.0
1,2026-07-08,22.3,14.0,0.0
2,2026-07-09,25.8,12.3,0.0
3,2026-07-10,22.4,12.2,0.0
4,2026-07-11,26.3,12.1,0.0


## Explore the Data Frame

In [8]:
# Display the basic information about the DataFrame.
weather_df.info()

# Display summary stattistics for the numeric column
weather_df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   time                7 non-null      str    
 1   temperature_2m_max  7 non-null      float64
 2   temperature_2m_min  7 non-null      float64
 3   precipitation_sum   7 non-null      float64
dtypes: float64(3), str(1)
memory usage: 356.0 bytes


,temperature_2m_max,temperature_2m_min,precipitation_sum
count,7.000000,7.000000,7.0
mean,25.214286,12.014286,0.0
std,2.896796,1.552878,0.0
min,22.300000,9.800000,0.0
25%,23.350000,11.150000,0.0
50%,24.600000,12.200000,0.0
75%,26.050000,12.900000,0.0
max,30.800000,14.000000,0.0


## Convert the Date Column

In [9]:
# Convert the time column from a string to a datetime object.
weather_df['time'] = pd.to_datetime(weather_df['time'])

# Verify that the data type has changed.
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   time                7 non-null      datetime64[us]
 1   temperature_2m_max  7 non-null      float64       
 2   temperature_2m_min  7 non-null      float64       
 3   precipitation_sum   7 non-null      float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 356.0 bytes


## Add cleaner weather columns

In [10]:
# Rename the columns to cleaner, database-friendly names.
weather_df = weather_df.rename(
    columns={
        "time": "weather_date",
        "temperature_2m_max": "max_temp_c",
        "temperature_2m_min": "min_temp_c",
        "precipitation_sum": "precipitation_mm"
    }
)

# Add Fahrenheit versions so the data is easier to understand for a U.S. audience.
weather_df['max_temp_f'] = (weather_df['max_temp_c'] * 9/5) + 32
weather_df['min_temp_f'] = (weather_df['min_temp_c'] * 9/5) + 32

# Add a city column so this can support multiple cities later.
weather_df['city'] = 'Bellevue'

# Display the cleaned DF
weather_df.head()

,weather_date,max_temp_c,min_temp_c,precipitation_mm,max_temp_f,min_temp_f,city
0,2026-07-07,24.6,13.5,0.0,76.28,56.30,Bellevue
1,2026-07-08,22.3,14.0,0.0,72.14,57.20,Bellevue
2,2026-07-09,25.8,12.3,0.0,78.44,54.14,Bellevue
3,2026-07-10,22.4,12.2,0.0,72.32,53.96,Bellevue
4,2026-07-11,26.3,12.1,0.0,79.34,53.78,Bellevue


## Check the cleaned data

In [11]:
# Confirm the column names, data types, and missing values.
weather_df.info()

# Count missing values in each column.
weather_df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   weather_date      7 non-null      datetime64[us]
 1   max_temp_c        7 non-null      float64       
 2   min_temp_c        7 non-null      float64       
 3   precipitation_mm  7 non-null      float64       
 4   max_temp_f        7 non-null      float64       
 5   min_temp_f        7 non-null      float64       
 6   city              7 non-null      str           
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 524.0 bytes


weather_date        0
max_temp_c          0
min_temp_c          0
precipitation_mm    0
max_temp_f          0
min_temp_f          0
city                0
dtype: int64

## Basic data quality checks

In [13]:
# Check that the DataFrame is not empty.
assert not weather_df.empty, "The DataFrame is empty!"

# Check that required columns are present.
required_columns= [
    "weather_date",
    "max_temp_c",
    "min_temp_c",
    "precipitation_mm",
    "max_temp_f",
    "min_temp_f",
    "city"
]

assert set(required_columns).issubset(weather_df.columns), "Missing required columns!"

# Check for missing values in important columns.
assert (weather_df['max_temp_c'] >= weather_df['min_temp_c']).all(), "Invalid temperature found."

print("All data quality checks passed!")

All data quality checks passed!


## Save cleaned Data

In [12]:
from pathlib import Path # Helps create file paths that work across operating systems.

# Define where the processed data will be saved.
processed_path = Path('../data/processed/bellevue_weather_daily.csv')

# Save the cleaned DataFrame to a CSV without the pandas index column.
weather_df.to_csv(processed_path, index=False)

# Confirm where the file was saved.
processed_path

WindowsPath('../data/processed/bellevue_weather_daily.csv')